# 大数据学期项目：RAG 检索增强生成流水线演示
## 方向 B：智能客户支持与检索增强生成（RAG）助手

> 本 Notebook 用于课堂/答辩现场演示项目代码库，按真实流水线展示：环境检查 → 数据摄取 → 清洗分块 → 元数据处理 → 向量索引 → 混合检索 → 答案生成 → 评估指标。
>
> 当前最终提交口径：**176 个原始 Markdown 文件**（其中空文件会在摄取时跳过）、**108 个 pytest 用例全部通过**、域内 **Recall@3 = 87.8%**，并已修复 ChromaDB 多键 `where` 需要显式 `$and` 的兼容问题。

## 0. 环境准备

运行前请确认项目根目录下存在 `.env`。本项目支持三种嵌入模式：

- `local`：默认模式，使用 `BAAI/bge-large-zh-v1.5` 本地/镜像模型。
- `remote`：用于百万级 ETL，连接 AutoDL RTX 4090 上的 OpenAI-compatible Embedding 服务。
- 其他模型名：作为 OpenAI-compatible 远程嵌入 API 调用。

`.env` 中的 API Key 和远程 Token 只用于现场运行，不会在本 Notebook 中明文输出。

In [2]:
# 依赖安装（首次运行时取消注释）
# !pip install -r requirements.txt

import os
from pathlib import Path

from dotenv import load_dotenv

BASE_DIR = Path.cwd()
env_path = BASE_DIR / ".env"

if not env_path.exists():
    print("未找到 .env，请先复制 .env.example 并填写必要配置：")
    print("  copy .env.example .env")
else:
    load_dotenv(env_path)
    api_key = os.getenv("OPENAI_API_KEY", "")
    emb_model = os.getenv("OPENAI_EMBEDDING_MODEL", "local")
    emb_base = os.getenv("OPENAI_EMBEDDING_BASE_URL", "")
    local_emb = os.getenv("LOCAL_EMBEDDING_MODEL", "BAAI/bge-large-zh-v1.5")
    token = os.getenv("EMBEDDING_SERVER_TOKEN", "")

    print("已加载 .env")
    print(f"  OPENAI_API_KEY: {'已配置' if api_key else '未配置'}")
    print(f"  对话模型: {os.getenv('OPENAI_MODEL', 'gpt-4o-mini')}")
    print(f"  嵌入模式: {emb_model or 'local'}")
    print(f"  本地/远程嵌入模型: {local_emb}")
    print(f"  远程嵌入服务: {emb_base or '未配置'}")
    print(f"  远程服务 Token: {'已配置' if token else '未配置'}")

已加载 .env
  OPENAI_API_KEY: 已配置
  对话模型: deepseek-v4-flash
  嵌入模式: local
  本地/远程嵌入模型: models/bge-large-zh-v1.5
  远程嵌入服务: http://localhost:6008/v1
  远程服务 Token: 未配置


## 1. 项目结构总览

项目按课程要求保持模块化边界：`src/` 放核心流水线，`app/` 放 Streamlit Web 入口，`tests/` 放离线自动化测试，`report/` 放最终报告与评估产物。

In [3]:
print("项目根目录:", BASE_DIR)
print()

SKIP_NAMES = {".git", ".venv", ".pytest_cache", "__pycache__", "vector_store", "models", "scratch"}

def show_tree(path: Path, prefix: str = "", max_depth: int = 2, current_depth: int = 0) -> None:
    if current_depth > max_depth:
        return
    items = [
        item for item in sorted(path.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))
        if item.name not in SKIP_NAMES
    ]
    for index, item in enumerate(items):
        connector = "└── " if index == len(items) - 1 else "├── "
        if item.is_dir():
            print(f"{prefix}{connector}{item.name}/")
            next_prefix = prefix + ("    " if index == len(items) - 1 else "│   ")
            show_tree(item, next_prefix, max_depth, current_depth + 1)
        else:
            print(f"{prefix}{connector}{item.name}")

show_tree(BASE_DIR, max_depth=2)

项目根目录: d:\date analysis\final exam2

├── .claude/
│   └── worktrees/
│       ├── adoring-margulis-d2c27f/
│       └── focused-napier-5a096f/
├── .vercel/
│   ├── project.json
│   └── README.txt
├── .vscode/
│   └── settings.json
├── app/
│   ├── rendering.py
│   ├── retrieval_fallback.py
│   ├── streamlit_app.py
│   └── style.css
├── artifacts/
│   ├── frontend-desktop.png
│   └── frontend-mobile.png
├── data/
│   └── raw/
│       ├── external/
│       ├── architecture_notes.md
│       ├── case_study_01.md
│       ├── case_study_02.md
│       ├── case_study_03.md
│       ├── case_study_04.md
│       ├── case_study_05.md
│       ├── case_study_06.md
│       ├── case_study_07.md
│       ├── case_study_08.md
│       ├── case_study_09.md
│       ├── case_study_10.md
│       ├── chunking_notes.md
│       ├── course_intro.md
│       ├── course_schedule.md
│       ├── dataset_guidelines.md
│       ├── demo_script.md
│       ├── embedding_notes.md
│       ├── evaluation_notes.md
│       ├── fa

## 2. 导入核心模块

这里直接导入项目源码中的流水线函数：

- `ingest.py`：读取 Markdown/TXT/PDF 与 JSONL。
- `preprocess.py`：清洗、分块、元数据融合。
- `embed_store.py`：ChromaDB 持久化、嵌入、混合检索。
- `query_parser.py`：自然语言查询解析。
- `qa.py`：基于检索结果生成带来源的答案。

In [4]:
from src.utils import init_env, get_openai_client, get_model_name, get_embedding_model_name
from src.ingest import load_text_files, load_jsonl_files
from src.preprocess import process_documents, clean_text, chunk_text
from src.embed_store import VectorStore
from src.qa import generate_answer
from src.query_parser import parse_query
from src.collect_corpus import TOPICS

init_env()
client = get_openai_client()
print(f"LLM 模型: {get_model_name()}")
print(f"Embedding 配置: {get_embedding_model_name()}")
print(f"客户端类型: {type(client).__name__}")

ModuleNotFoundError: No module named 'openai'

## 3. 数据源一览

当前仓库中的 `data/raw/` 是演示用可运行语料，包含课程资料、FAQ、通知、技术词条，以及已采集的外部资料。

In [5]:
data_dir = BASE_DIR / "data" / "raw"
md_files = list(data_dir.rglob("*.md"))
txt_files = list(data_dir.rglob("*.txt"))
pdf_files = list(data_dir.rglob("*.pdf"))
external_dir = data_dir / "external"

wiki_files = list(external_dir.glob("wiki_*.md")) if external_dir.exists() else []
so_files = list(external_dir.glob("so_*.md")) if external_dir.exists() else []
csdn_files = list(external_dir.glob("csdn_*.md")) if external_dir.exists() else []
course_files = [p for p in md_files if external_dir not in p.parents]

print(f"Markdown 文件: {len(md_files)} 个")
print(f"TXT 文件:      {len(txt_files)} 个")
print(f"PDF 文件:      {len(pdf_files)} 个")
print(f"课程/自建资料: {len(course_files)} 个")
print(f"Wikipedia:    {len(wiki_files)} 篇")
print(f"Stack Overflow: {len(so_files)} 篇")
print(f"CSDN:         {len(csdn_files)} 篇")
print(f"原始文件总数: {len(md_files) + len(txt_files) + len(pdf_files)} 个")
print("说明: 空文件会在摄取阶段跳过，实际加载数量以下一节 load_text_files() 为准")
print("报告中的百万级主索引规模: 1,215,021 个文本分块；当前本地集合数量以 store.count() 为准")

Markdown 文件: 176 个
TXT 文件:      0 个
PDF 文件:      0 个
课程/自建资料: 45 个
Wikipedia:    83 篇
Stack Overflow: 30 篇
CSDN:         18 篇
原始文件总数: 176 个
说明: 空文件会在摄取阶段跳过，实际加载数量以下一节 load_text_files() 为准
报告中的百万级主索引规模: 1,215,021 个文本分块；当前本地集合数量以 store.count() 为准


### 3.1 多源语料采集

| 来源 | 模块 | 内容 |
|------|------|------|
| Wikipedia | `collect_corpus.py` | 大数据、机器学习、RAG 等技术术语摘要 |
| Stack Overflow | `collect_stackoverflow.py` | 高票技术问答，清洗为中文 Markdown |
| CSDN 博客 | `collect_csdn.py` | 中文技术博客，使用 BeautifulSoup + html2text 去噪 |
| 高级采集 | `collect_more_corpus.py` | 自适应高级语料补充 |

一键采集命令：`python src/main.py collect-all`。

In [6]:
print(f"Wikipedia 预置主题 ({len(TOPICS)} 个):")
for index, topic in enumerate(TOPICS, 1):
    print(f"  {index:2d}. {topic.zh_title} ({topic.en_title}) [{topic.tag}]")
    if index >= 10:
        print(f"  ... 还有 {len(TOPICS) - 10} 个主题")
        break

print("\n已采集外部语料:")
print(f"  Wikipedia 词条: {len(wiki_files)} 篇")
print(f"  Stack Overflow 问答: {len(so_files)} 篇")
print(f"  CSDN 博客: {len(csdn_files)} 篇")

NameError: name 'TOPICS' is not defined

## 4. 数据摄取

`load_text_files()` 负责读取 Markdown/TXT/PDF；`load_jsonl_files()` 负责读取大规模 JSONL 语料。二者输出统一字典结构：

```python
{"source": str, "path": str, "text": str, "fm_meta": dict}
```

这样后续清洗、分块和建库不需要关心原始文件格式。

In [7]:
documents = load_text_files(data_dir)
jsonl_documents = load_jsonl_files(BASE_DIR / "data")
all_documents = documents + jsonl_documents

print(f"Markdown/TXT/PDF 文档: {len(documents)} 篇")
print(f"JSONL 文档:          {len(jsonl_documents)} 篇")
print(f"摄取文档总数:        {len(all_documents)} 篇")

if all_documents:
    sample = all_documents[0]
    print("\n示例文档:")
    print(f"  source: {sample['source']}")
    print(f"  path: {sample['path']}")
    print(f"  text 长度: {len(sample['text'])} 字符")
    print(f"  fm_meta: {sample.get('fm_meta', {})}")

NameError: name 'load_text_files' is not defined

## 5. 数据清洗与语义分块

`clean_text()` 的清洗流程：移除 HTML 标签 → 解码 HTML 实体 → 过滤控制字符 → 规范化空白。

`chunk_text()` 采用四层优先级策略，尽量保护段落、句子和代码块边界：

```text
段落边界 → 句子边界 → 贪心合并 → 滑窗切割
```

默认参数为 `chunk_size=700`、`overlap=120`。

In [8]:
sample_doc = documents[0]
sample_cleaned = clean_text(sample_doc["text"])
chunks = chunk_text(sample_cleaned, chunk_size=700, overlap=120)

print(f"文档: {sample_doc['source']}")
print(f"原始长度: {len(sample_doc['text'])} 字符")
print(f"清洗后长度: {len(sample_cleaned)} 字符")
print(f"分块数: {len(chunks)}")

for index, chunk in enumerate(chunks[:5], 1):
    preview = chunk["text"][:80].replace("\n", " ")
    print(f"块 {index} [{chunk['char_start']}:{chunk['char_end']}]: {preview}...")
if len(chunks) > 5:
    print(f"... 还有 {len(chunks) - 5} 个分块")

NameError: name 'documents' is not defined

## 6. 元数据与预处理流水线

`process_documents()` 把摄取得到的文档转换为可写入向量库的分块列表。

- `metadata_strategy="merge"`：默认策略，人工/结构化元数据优先，LLM 补充缺失字段。
- `metadata_strategy="jsonl_only"`：百万级 JSONL 建库时可跳过 LLM 元数据提取，降低成本。
- 输出结构为 `{"id": str, "text": str, "metadata": {...}}`。

现场演示默认使用 `is_extract_meta=False`，避免重复调用 LLM。

In [9]:
import json

print("开始预处理文档（is_extract_meta=False，避免现场重复调用 LLM）...")
processed = process_documents(
    all_documents,
    chunk_size=700,
    overlap=120,
    is_extract_meta=False,
    metadata_strategy="merge",
)

print("\n预处理完成")
print(f"输入文档: {len(all_documents)} 篇")
print(f"输出分块: {len(processed)} 个")
print(f"平均每篇分块: {len(processed) / max(len(all_documents), 1):.1f}")

print("\n分块样例:")
preview = {key: str(value)[:120] for key, value in processed[0].items()}
print(json.dumps(preview, indent=2, ensure_ascii=False))

开始预处理文档（is_extract_meta=False，避免现场重复调用 LLM）...


NameError: name 'process_documents' is not defined

## 7. 向量索引与 ChromaDB

`VectorStore` 使用 ChromaDB 本地持久化目录 `vector_store/`，默认集合名为 `course_docs`，距离度量为 cosine。

当前嵌入支持：

- 本地/镜像模型：`BAAI/bge-large-zh-v1.5`（当前 `.env` 可指向 `models/bge-large-zh-v1.5` 本地目录）。
- 远程 GPU 服务：AutoDL RTX 4090 + FastAPI OpenAI-compatible `/v1/embeddings`。
- OpenAI-compatible API：适配其他远程嵌入服务。

In [10]:
store = VectorStore(collection_name="course_docs")
print(f"集合名称: {store.collection.name}")
print(f"当前向量库分块数: {store.count()}（以本机 ChromaDB 实际状态为准）")
print(f"嵌入模型: {store.embedding_model}")
print("距离度量: Cosine (HNSW)")

NameError: name 'VectorStore' is not defined

## 8. 检索：语义搜索 + 元数据过滤

`VectorStore.search()` 是外部调用入口，内部复用 `hybrid_search()`：

1. 纯语义检索：按向量相似度返回 Top-K。
2. 元数据过滤：支持 `where={"category": "wiki"}` 等条件。
3. 多键过滤兼容：当 `where` 同时包含多个键时，自动转换为显式 `{"$and": [...]}`，适配 ChromaDB。
4. 距离阈值：`max_distance` 可截断低相关结果，降低幻觉风险。

In [11]:
query = "什么是检索增强生成"
print(f"查询: {query}")

try:
    results = store.search(query, top_k=3)
    print(f"返回条数: {len(results)}")
    for index, item in enumerate(results, 1):
        score = item.get("score")
        similarity = max(0, (1 - score / 2)) * 100 if score is not None else 0
        preview = item["text"][:100].replace("\n", " ")
        print(f"[{index}] {item['source']} (距离={score:.3f}, 相似度={similarity:.0f}%)")
        print(f"    内容: {preview}...")
except Exception as exc:
    if isinstance(exc, (KeyboardInterrupt, SystemExit)):
        raise
    results = []
    print(f"检索演示跳过: {exc}")

查询: 什么是检索增强生成
检索演示跳过: name 'store' is not defined


In [12]:
print("元数据过滤示例：多键 where 自动转为显式 $and")
try:
    multi_filter = {"category": "notice", "year": 2024}
    results_multi = store.search("2024 年的课程通知", top_k=3, where=multi_filter)
    print(f"过滤条件: {multi_filter}")
    print(f"返回结果: {len(results_multi)} 条")
    for item in results_multi:
        meta = item.get("metadata", {}) or {}
        print(f"  {item['source']} | 分类={meta.get('category')} | 年份={meta.get('year')}")
except Exception as exc:
    if isinstance(exc, (KeyboardInterrupt, SystemExit)):
        raise
    print(f"过滤检索演示跳过: {exc}")

元数据过滤示例：多键 where 自动转为显式 $and
过滤检索演示跳过: name 'store' is not defined


## 9. 查询意图解析与答案生成

`query_parser.py` 使用 LLM 将自然语言问题解析为：

```python
{"search_query": str, "filters": dict | None, "raw_filters": dict}
```

`qa.py` 将检索结果注入 LLM，要求答案必须基于参考资料，并带来源标注。

In [13]:
test_queries = [
    "2024年的课程通知里说了什么？",
    "RAG架构中向量数据库的作用是什么？",
    "去年老师有没有说过期末怎么考？",
]

for question in test_queries:
    try:
        parsed = parse_query(question, client=client)
    except Exception as exc:
        if isinstance(exc, (KeyboardInterrupt, SystemExit)):
            raise
        parsed = {"search_query": question, "filters": None, "raw_filters": {}}
        print(f"解析失败，使用回退结构: {exc}")

    print(f"用户输入: {question}")
    print(f"  搜索词: {parsed['search_query']}")
    print(f"  过滤条件: {parsed['filters']}")
    print()

解析失败，使用回退结构: name 'parse_query' is not defined
用户输入: 2024年的课程通知里说了什么？
  搜索词: 2024年的课程通知里说了什么？
  过滤条件: None

解析失败，使用回退结构: name 'parse_query' is not defined
用户输入: RAG架构中向量数据库的作用是什么？
  搜索词: RAG架构中向量数据库的作用是什么？
  过滤条件: None

解析失败，使用回退结构: name 'parse_query' is not defined
用户输入: 去年老师有没有说过期末怎么考？
  搜索词: 去年老师有没有说过期末怎么考？
  过滤条件: None



In [14]:
question = "Spark和Kafka有什么区别？"
print(f"问题: {question}\n")

try:
    parsed = parse_query(question, client=client)
    print(f"解析出的搜索词: {parsed['search_query']}")
    print(f"解析出的过滤: {parsed['filters']}\n")

    retrieved = store.search(parsed["search_query"], top_k=3, where=parsed["filters"])
    print(f"检索到 {len(retrieved)} 条相关文档:")
    for index, item in enumerate(retrieved, 1):
        print(f"  [{index}] {item['source']} (距离={item['score']:.3f})")

    answer = generate_answer(question, retrieved, client=client)
    print("\n" + "=" * 60)
    print("答案:")
    print(answer)
    print("=" * 60)
except Exception as exc:
    if isinstance(exc, (KeyboardInterrupt, SystemExit)):
        raise
    print(f"答案生成演示跳过: {exc}")

问题: Spark和Kafka有什么区别？

答案生成演示跳过: name 'parse_query' is not defined


## 10. 防幻觉机制

| 层级 | 措施 |
|------|------|
| System Prompt | 要求仅根据参考资料回答 |
| 上下文限制 | 只把检索结果作为参考资料传入 LLM |
| 来源检查 | 若回答缺少来源，自动追加来源列表 |
| 距离阈值 | `max_distance` 可过滤低相关召回块 |
| 超纲拒答 | 知识库无依据时提示无法基于资料回答 |

最终评估口径：自动评估中超纲查询幻觉标记为 **4/9**；人工复核中真正实质性编造为 **1/9**，其余主要是“已拒答但引用了无关来源”的严格脚本判定。

In [15]:
unrelated = "钢琴考级需要准备什么？"
print(f"超纲问题: {unrelated}")

try:
    parsed = parse_query(unrelated, client=client)
    retrieved = store.search(parsed["search_query"], top_k=3, max_distance=1.0)
    answer = generate_answer(unrelated, retrieved, client=client)
    print(f"检索到 {len(retrieved)} 条")
    print(f"答案:\n{answer}")
except Exception as exc:
    if isinstance(exc, (KeyboardInterrupt, SystemExit)):
        raise
    print(f"超纲拒答演示跳过: {exc}")

超纲问题: 钢琴考级需要准备什么？
超纲拒答演示跳过: name 'parse_query' is not defined


## 11. 系统状态与评估指标

最终报告中的关键评估数据来自 `report/evaluation_results.json` 与 `report/latency_results.json`。这里直接读取文件，避免 notebook 和报告口径不一致。

In [16]:
print("=" * 40)
print("  系统状态面板")
print("=" * 40)
print(f"  向量库集合: {store.collection.name}")
print(f"  文档块总数: {store.count()}")
print(f"  来源文件数: {len(store.list_sources())}")
print(f"  LLM 模型:   {get_model_name()}")
print(f"  Embedding:  {store.embedding_model}")
print(f"  嵌入模式:   {'本地模型' if store._use_local else ('远程 GPU' if store._use_remote else 'OpenAI-compatible API')}")

  系统状态面板


NameError: name 'store' is not defined

In [17]:
import json

with open(BASE_DIR / "report" / "evaluation_results.json", "r", encoding="utf-8") as file:
    evaluation = json.load(file)["summary"]

with open(BASE_DIR / "report" / "latency_results.json", "r", encoding="utf-8") as file:
    latency = json.load(file)["summary"]

print("检索与回答评估:")
print(f"  测试查询总数: {evaluation['total_queries']}")
print(f"  域内查询: {evaluation['in_scope_queries']}")
print(f"  超纲查询: {evaluation['out_of_scope_queries']}")
print(f"  Recall@3: {evaluation['recall_at_3'] * 100:.1f}% ({evaluation['in_scope_hit_count']}/{evaluation['in_scope_queries']})")
print(f"  自动评估幻觉标记: {evaluation['hallucination_count']}/{evaluation['out_of_scope_queries']}")
print(f"  超纲拒答准确率: {evaluation['out_scope_accuracy'] * 100:.2f}%")
print(f"  评估平均延迟: {evaluation['avg_latency_s']:.3f}s")

print("\n独立延迟基准:")
print(f"  查询解析: {latency['avg_parse_s']:.3f}s")
print(f"  向量检索: {latency['avg_search_s']:.3f}s")
print(f"  答案生成: {latency['avg_generate_s']:.3f}s")
print(f"  端到端: {latency['avg_total_s']:.3f}s")
print(f"  瓶颈阶段: {latency['bottleneck']}")

检索与回答评估:
  测试查询总数: 50
  域内查询: 41
  超纲查询: 9
  Recall@3: 87.8% (36/41)
  自动评估幻觉标记: 4/9
  超纲拒答准确率: 55.56%
  评估平均延迟: 5.637s

独立延迟基准:
  查询解析: 2.313s
  向量检索: 3.430s
  答案生成: 3.081s
  端到端: 8.824s
  瓶颈阶段: search


## 12. CLI、Web、AutoDL 与公网演示运行方式

### CLI

```bash
# 采集语料
python src/main.py collect          # Wikipedia
python src/main.py collect-so       # Stack Overflow
python src/main.py collect-csdn     # CSDN
python src/main.py collect-all      # 全量采集

# 建立/更新索引
python src/main.py build --metadata-strategy merge
python src/main.py build --metadata-strategy jsonl_only  # 大规模 JSONL 可跳过 LLM 元数据提取

# 问答
python src/main.py ask --question "课程项目提交要求是什么？"
python src/main.py ask              # 交互模式
```

### 本地 Web

```bash
streamlit run app/streamlit_app.py
```

### Cloudflare Tunnel 公网演示

本项目已新增一键公网演示脚本。该方案不上传 9GB+ 向量库和本地模型，只让 Cloudflare Tunnel 把本机正在运行的 Streamlit 页面安全映射到临时 HTTPS 地址，适合答辩现场或给同学远程试用。

```powershell
# 启动 Streamlit + Cloudflare Quick Tunnel，控制台会打印 trycloudflare.com 公网地址
powershell -NoProfile -ExecutionPolicy Bypass -File .\scripts\start_public_streamlit.ps1

# 只检查/启动本地 Streamlit，不启动公网 Tunnel
powershell -NoProfile -ExecutionPolicy Bypass -File .\scripts\start_public_streamlit.ps1 -NoTunnel

# 停止 Tunnel 与当前 8502 端口的 Streamlit
powershell -NoProfile -ExecutionPolicy Bypass -File .\scripts\stop_public_streamlit.ps1
```

> Quick Tunnel 地址是临时随机域名，关闭电脑或重启 Tunnel 后会变化；若要固定域名，应在 Cloudflare Zero Trust 中创建 named tunnel 并绑定自己的域名。

### Vercel 静态入口

当前 Vercel Hobby 部署用于项目介绍页和入口跳转，不承载 Streamlit、ChromaDB、向量库或本地 embedding 模型。完整交互仍由本地 Streamlit + Cloudflare Tunnel 提供。

- 生产入口：https://final-exam2-rag.vercel.app
- 部署配置：`index.html`、`vercel.json`、`.vercelignore`

### AutoDL 远程 Embedding

```bash
# 远程服务端：启动 FastAPI Embedding 服务
EMBEDDING_SERVER_TOKEN=<your-token> bash scripts/setup_autodl.sh

# 本地 .env
OPENAI_EMBEDDING_MODEL=remote
OPENAI_EMBEDDING_BASE_URL=https://<autodl-host>/v1
EMBEDDING_SERVER_TOKEN=<your-token>
LOCAL_EMBEDDING_MODEL=BAAI/bge-large-zh-v1.5
```

## 13. 项目技术总结

| 模块 | 技术选择 | 说明 |
|------|----------|------|
| 语言 | Python 3.11+ | 课程要求与生态兼容 |
| 数据摄取 | Markdown/TXT/PDF/JSONL | 支持课程文档和百万级 JSONL |
| 清洗分块 | `clean_text()` + 四层语义分块 | 尽量保护段落、句子、FAQ 和代码块边界 |
| 元数据 | Front-Matter/JSONL 优先 + LLM 补充 | 降低 API 成本并提高可控性 |
| 向量库 | ChromaDB | 本地持久化、部署简单、适合课程演示 |
| Embedding | `BAAI/bge-large-zh-v1.5` | 1024 维中文语义表征，支持本地/远程 GPU |
| 检索 | 向量搜索 + 元数据过滤 + `$and` 兼容 | 支持自然语言问题和结构化约束 |
| 问答 | OpenAI-compatible LLM | 带来源引用，支持拒答 |
| Web 交互 | Streamlit + Cloudflare Tunnel | 本机保留模型/向量库，公网只转发交互页面 |
| 静态入口 | Vercel Hobby | 项目介绍页和跳转入口，不承载重后端 |
| 测试 | pytest + unittest.mock.patch | 108 个用例，覆盖离线单测、回退与安全渲染 |
| 报告 | `report/report_ieee.html` / `.pdf` | IEEE 风格最终报告与可打印交付物 |

---

**最终交付文件参考**：

- 项目入口：`src/main.py`、`app/streamlit_app.py`
- 公网演示脚本：`scripts/start_public_streamlit.ps1`、`scripts/stop_public_streamlit.ps1`
- Vercel 静态入口：`index.html`、`vercel.json`、`.vercelignore`
- 演示 Notebook：`pipeline_demo.ipynb`
- 最终报告：`report/report_ieee.html`、`report/report_ieee.pdf`
- 评估结果：`report/evaluation_results.json`、`report/latency_results.json`
- 自动化测试：`tests/`（当前 108 个用例全量通过）

**现场演示建议**：先运行环境检查、数据摄取、清洗分块、系统状态和评估指标；如果网络/API 状态稳定，再运行查询解析与答案生成单元。需要远程演示时，优先使用 `scripts/start_public_streamlit.ps1` 获取临时 HTTPS 地址；如果网络不稳，再切换到本地页面或录制视频兜底。